# The library checkout that went wrong
### Transactions and isolation, one small experiment at a time

Imagine lending a book: the library must mark the **copy** as borrowed *and* record **who borrowed it**. If only one part is saved, the records no longer tell the same story.

**Today:** see a real rollback, watch two connections read the same data, and connect the idea to PostgreSQL and AI systems.

**To follow along:** run each cell from top to bottom with **Shift + Enter**. Predict the result before each code cell. This notebook uses Python's built-in **SQLite** and a temporary file; no installation or real member information is needed. SQLite is used to teach the idea—it is not PostgreSQL. To reset the experiment, run all cells from the top.

## 1 · Set the scene

We create a tiny database with one member, two copies and an empty loan table. A **connection** is like a separate librarian using the same database. `a` and `b` are two independent connections.

A **foreign key** is a safety rule: a loan may refer only to a member who exists. We deliberately use this rule to trigger a failure later. WAL is a SQLite setting that lets our second connection read while the first has an unfinished write.

In [ ]:
import sqlite3
import tempfile
from pathlib import Path

# If we run the notebook again, close the previous demo first.
for name in ("a", "b"):
    if name in globals():
        globals()[name].close()
if "folder" in globals():
    folder.cleanup()

folder = tempfile.TemporaryDirectory()  # A disposable place for our demo.
file = Path(folder.name) / "library.db"
a = sqlite3.connect(file, isolation_level=None)
b = sqlite3.connect(file, isolation_level=None)
a.execute("PRAGMA foreign_keys = ON")
b.execute("PRAGMA foreign_keys = ON")
a.execute("PRAGMA journal_mode = WAL")

a.executescript("""
    CREATE TABLE members (id INTEGER PRIMARY KEY);
    CREATE TABLE copies (id INTEGER PRIMARY KEY, status TEXT NOT NULL);
    CREATE TABLE loans (copy_id INTEGER, member_id INTEGER REFERENCES members(id));
    INSERT INTO members VALUES (7);
    INSERT INTO copies VALUES (101, 'available'), (102, 'available');
""")
print("Ready: member 7, copies 101 and 102.")

## 2 · Look before changing anything

A SQL `SELECT` reads data. `?` is a placeholder for a value; passing `(copy_id,)` separately is safer and clearer than building SQL with string concatenation.

`show` prints a copy's status and how many loans are recorded for it. Both copies start **available**, with **zero** loans.

In [ ]:
def show(connection, copy_id):
    """Show one copy and its recorded loans."""
    status = connection.execute(
        "SELECT status FROM copies WHERE id = ?", (copy_id,)
    ).fetchone()[0]
    count = connection.execute(
        "SELECT COUNT(*) FROM loans WHERE copy_id = ?", (copy_id,)
    ).fetchone()[0]
    print(f"Copy {copy_id}: {status}; loans: {count}")

show(a, 101)
show(b, 102)

## 3 · Break a checkout on purpose

A **transaction** is a group of database changes treated as one unit:

1. `BEGIN` starts the group.
2. We mark copy 101 as loaned.
3. We try to add a loan for member **999**. That member does not exist, so the database rejects it.
4. `ROLLBACK` undoes the earlier, unfinished update.

**Predict:** After the rollback, will copy 101 still be marked loaned?

In [ ]:
a.execute("BEGIN")
try:
    a.execute("UPDATE copies SET status = 'loaned' WHERE id = 101")
    print("Inside the transaction:")
    show(a, 101)
    a.execute("INSERT INTO loans VALUES (101, 999)")  # Member 999 is missing.
    a.execute("COMMIT")
except sqlite3.IntegrityError:
    print("The loan failed: member 999 does not exist.")
    a.execute("ROLLBACK")

print("After rollback:")
show(a, 101)

### What happened? Now try a successful checkout

Inside the transaction, connection `a` could see its own unfinished update. After the failed insert and explicit rollback, the copy is available again and there is no loan. This **all-or-nothing** property is called **atomicity**.

In this SQLite example, the foreign-key error normally undoes the failed statement—not the earlier update. The explicit `ROLLBACK` undoes the whole unfinished transaction. PostgreSQL handles statement errors differently: an error normally leaves the transaction failed until a rollback or appropriate savepoint recovery.

Now member 7 borrows copy 101. We update the copy and add the loan, then `COMMIT` saves both. The `WHERE` condition and row-count check make this demo refuse a copy that is already marked loaned. A real service also needs database constraints and concurrency-safe logic. If an error occurs, we roll back and re-raise it rather than pretending the checkout succeeded.

In [ ]:
a.execute("BEGIN")
try:
    changed = a.execute(
        "UPDATE copies SET status = 'loaned' WHERE id = 101 AND status = 'available'"
    )
    if changed.rowcount != 1:
        raise ValueError("Copy is not available")
    a.execute("INSERT INTO loans VALUES (101, 7)")
    a.execute("COMMIT")
except Exception:
    a.execute("ROLLBACK")
    raise

show(b, 101)  # The other connection sees the committed checkout.

## 4 · What can another librarian see?

Now `a` begins a checkout for copy 102 but pauses before committing. While it is paused, `b` reads the copy. Then `a` commits and `b` reads again.

**Predict:** Does `b` see the unfinished change? `b` is in autocommit mode, so its later read starts a new read transaction. This demonstration uses SQLite's WAL mode; other database systems have their own rules.

In [ ]:
a.execute("BEGIN")
a.execute("UPDATE copies SET status = 'loaned' WHERE id = 102")
a.execute("INSERT INTO loans VALUES (102, 7)")

print("Before commit, connection B sees:")
show(b, 102)
a.execute("COMMIT")
print("After commit, a new read by B sees:")
show(b, 102)

**Isolation** controls how concurrent transactions interact. Here, `b` saw the previously committed state before the commit and the new state on a later read. That is not a dirty read. This example does **not** prove that every race is prevented.

### From this demo to PostgreSQL and AI engineering

- PostgreSQL **READ COMMITTED** normally gives each statement a fresh snapshot; two reads in one transaction can see different commits. **REPEATABLE READ** keeps a transaction snapshot, but can still allow write skew. **SERIALIZABLE** can abort one transaction; retry the **whole** transaction when appropriate.
- Two people requesting the last copy—or two workers claiming the same embedding job—need a concurrency-safe write and database constraints, not only an earlier `SELECT`.
- A database rollback cannot unsend an email, reverse a model-provider call or undo a vector-store write. For reliable external work, consider a **transactional outbox**, a retryable worker and idempotent consumers.
- An **idempotency key** can prevent a retried request from creating the same logical work twice, if the key and result are stored atomically. A unique constraint can help enforce that rule.

These are design ideas, not guarantees supplied by this short SQLite demo. PostgreSQL locking, errors and isolation behave differently; test your actual PostgreSQL implementation.

## 5 · Check the result yourself

Before running the next cell, answer aloud:

1. Which command undoes an unfinished transaction?
2. What did connection `b` see before the second checkout committed?
3. Why is an external model call not automatically undone by database rollback?

This final cell checks the database state we expect after the two successful demonstrations. An `assert` stops with an error if a check is false.

In [ ]:
status_101 = b.execute("SELECT status FROM copies WHERE id = 101").fetchone()[0]
status_102 = b.execute("SELECT status FROM copies WHERE id = 102").fetchone()[0]
loans = b.execute("SELECT copy_id, COUNT(*) FROM loans GROUP BY copy_id ORDER BY copy_id").fetchall()

assert status_101 == "loaned"
assert status_102 == "loaned"
assert loans == [(101, 1), (102, 1)]
print("All checks passed: each copy has one recorded checkout.")

### Answers and takeaway

1. `ROLLBACK` undoes the transaction's **uncommitted database work**.
2. It saw copy 102 as **available**, with no loan, before `a` committed.
3. A model-provider call happens outside the database transaction; the database cannot reverse it.

**Atomicity:** all database changes in the unit succeed or are undone. **Isolation:** rules for interacting with concurrent work. **Commit:** make changes permanent. **Rollback:** undo unfinished database changes.

The separate quiz contains harder, interview-style PostgreSQL and AI-system scenarios. It goes beyond the short code demo.